In [2]:
!pip install textblob vaderSentiment tabulate scikit-learn

In [3]:
# Import required libraries
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tabulate import tabulate
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
# Create sample data consisting of text samples with actual sentiment labels
data = [
    ("I love this product, it's amazing!", 'positive'),
    ("This product is terrible, I hate it.", 'negative'),
    ("It's okay, not bad but not great either.", 'neutral'),
    ("Best product ever, highly recommended!", 'positive'),
    ("I'm really disappointed with the quality.", 'negative'),
    ("So-so product, nothing special about it.", 'neutral'),
    ("The customer service was excellent!", 'positive'),
    ("I wasted my money on this useless product.", 'negative'),
    ("It's not the worst, but certainly not the best.", 'neutral'),
    ("I can't live without this product, it's a lifesaver!", 'positive'),
    ("The product arrived damaged and unusable.", 'negative'),
    ("It's average, neither good nor bad.", 'neutral'),
    ("Highly disappointed with the purchase.", 'negative'),
    ("The product exceeded my expectations.", 'positive'),
    ("It's just okay, nothing extraordinary.", 'neutral'),
    ("This product is excellent, it exceeded all my expectations!", 'positive'),
    ("I regret purchasing this product, it's a waste of money.", 'negative'),
    ("It's neither good nor bad, just average.", 'neutral'),
    ("Outstanding customer service, highly recommended!", 'positive'),
    ("I'm very disappointed with the quality of this item.", 'negative'),
    ("It's not the best product, but it gets the job done.", 'neutral'),
    ("This product is a game-changer, I can't imagine life without it!", 'positive'),
    ("I received a defective product, very dissatisfied.", 'negative'),
    ("It's neither great nor terrible, just okay.", 'neutral'),
    ("Fantastic product, I would buy it again in a heartbeat!", 'positive'),
    ("Avoid this product at all costs, complete waste of money.", 'negative'),
    ("It's decent, but nothing extraordinary.", 'neutral'),
    ("Impressive quality, exceeded my expectations!", 'positive'),
    ("I'm very unhappy with this purchase, total disappointment.", 'negative'),
    ("It's neither amazing nor terrible, somewhere in between.", 'neutral')
]

print(f"Total samples: {len(data)}")
print("Sample data created successfully!")

Total samples: 30
Sample data created successfully!


In [5]:
# Initialize an empty list to store the data in tabular format
table_data = [["Text", "Actual Label", "TextBlob Polarity", "TextBlob Sentiment", "VADER Compound", "VADER Sentiment"]]

print("Table initialized for results")

Table initialized for results


In [6]:
# Loop through each text and analyze sentiment
for text, actual_label in data:
    # TextBlob analysis
    blob = TextBlob(text)
    tb_polarity = blob.sentiment.polarity
    
    # Determine label based on polarity score from TextBlob
    if tb_polarity > 0:
        tb_label = 'positive'
    elif tb_polarity < 0:
        tb_label = 'negative'
    else:
        tb_label = 'neutral'
    
    # VADER analysis
    analyzer = SentimentIntensityAnalyzer()
    vs = analyzer.polarity_scores(text)
    vader_compound = vs['compound']
    
    # Determine label based on compound score from VADER
    if vader_compound > 0.05:
        vader_label = 'positive'
    elif vader_compound < -0.05:
        vader_label = 'negative'
    else:
        vader_label = 'neutral'
    
    # Add results to table
    table_data.append([text, actual_label, tb_polarity, tb_label, vader_compound, vader_label])

print("Sentiment analysis completed for all samples!")

Sentiment analysis completed for all samples!


In [7]:
# Print the sentiment analysis results in a table format
print("\n" + "="*120)
print("SENTIMENT ANALYSIS RESULTS: TEXTBLOB vs VADER")
print("="*120)
print(tabulate(table_data, headers="firstrow", tablefmt="grid"))


SENTIMENT ANALYSIS RESULTS: TEXTBLOB vs VADER
+------------------------------------------------------------------+----------------+---------------------+----------------------+------------------+-------------------+
| Text                                                             | Actual Label   |   TextBlob Polarity | TextBlob Sentiment   |   VADER Compound | VADER Sentiment   |
+==================================================================+================+=====================+======================+==================+===================+
| I love this product, it's amazing!                               | positive       |           0.625     | positive             |           0.8516 | positive          |
+------------------------------------------------------------------+----------------+---------------------+----------------------+------------------+-------------------+
| This product is terrible, I hate it.                             | negative       |          -0.9    

In [8]:
# Extract actual labels and predicted labels
actual_labels = [label for _, label, _, _, _, _ in table_data[1:]]
tb_predicted = [tb_label for _, _, _, tb_label, _, _ in table_data[1:]]
vader_predicted = [vader_label for _, _, _, _, _, vader_label in table_data[1:]]

# Calculate classification report for TextBlob
tb_classification_report = classification_report(
    actual_labels, 
    tb_predicted, 
    target_names=['negative', 'neutral', 'positive']
)

# Calculate classification report for VADER
vader_classification_report = classification_report(
    actual_labels, 
    vader_predicted, 
    target_names=['negative', 'neutral', 'positive']
)

# Print classification report for TextBlob
print("\n" + "="*60)
print("CLASSIFICATION REPORT FOR TEXTBLOB")
print("="*60)
print(tb_classification_report)

# Print classification report for VADER
print("\n" + "="*60)
print("CLASSIFICATION REPORT FOR VADER")
print("="*60)
print(vader_classification_report)


CLASSIFICATION REPORT FOR TEXTBLOB
              precision    recall  f1-score   support

    negative       0.67      0.80      0.73        10
     neutral       0.00      0.00      0.00        10
    positive       0.53      0.80      0.64        10

    accuracy                           0.53        30
   macro avg       0.40      0.53      0.46        30
weighted avg       0.40      0.53      0.46        30


CLASSIFICATION REPORT FOR VADER
              precision    recall  f1-score   support

    negative       0.56      1.00      0.71        10
     neutral       0.33      0.10      0.15        10
    positive       0.89      0.80      0.84        10

    accuracy                           0.63        30
   macro avg       0.59      0.63      0.57        30
weighted avg       0.59      0.63      0.57        30



In [9]:
# Extract texts and labels for machine learning
texts = [text for text, _ in data]
labels = [label for _, label in data]

# Split data into training and testing sets (60% train, 40% test)
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.4, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")
print(f"\nTraining set distribution:")
print(f"Positive: {y_train.count('positive')}")
print(f"Negative: {y_train.count('negative')}")
print(f"Neutral: {y_train.count('neutral')}")

Training set size: 18
Testing set size: 12

Training set distribution:
Positive: 5
Negative: 6
Neutral: 7


In [10]:
# Extract features (bag of words representation)
vectorizer = CountVectorizer()
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print(f"Feature matrix shape (training): {X_train_vectorized.shape}")
print(f"Feature matrix shape (testing): {X_test_vectorized.shape}")
print(f"Number of features: {len(vectorizer.get_feature_names_out())}")

Feature matrix shape (training): (18, 72)
Feature matrix shape (testing): (12, 72)
Number of features: 72


In [11]:
# Initialize classifiers
nb_classifier = MultinomialNB()
svm_classifier = SVC(kernel='linear')

# Train classifiers
print("Training Naive Bayes classifier...")
nb_classifier.fit(X_train_vectorized, y_train)
print("Naive Bayes training complete!")

print("\nTraining SVM classifier...")
svm_classifier.fit(X_train_vectorized, y_train)
print("SVM training complete!")

Training Naive Bayes classifier...
Naive Bayes training complete!

Training SVM classifier...
SVM training complete!


In [12]:
# Calculate classification report for Naive Bayes
nb_predictions = nb_classifier.predict(X_test_vectorized)
nb_classification_report = classification_report(
    y_test, 
    nb_predictions, 
    target_names=['negative', 'neutral', 'positive']
)

# Calculate classification report for SVM
svm_predictions = svm_classifier.predict(X_test_vectorized)
svm_classification_report = classification_report(
    y_test, 
    svm_predictions, 
    target_names=['negative', 'neutral', 'positive']
)

# Print classification report for Naive Bayes
print("\n" + "="*60)
print("CLASSIFICATION REPORT FOR NAIVE BAYES")
print("="*60)
print(nb_classification_report)

# Print classification report for SVM
print("\n" + "="*60)
print("CLASSIFICATION REPORT FOR SVM")
print("="*60)
print(svm_classification_report)


CLASSIFICATION REPORT FOR NAIVE BAYES
              precision    recall  f1-score   support

    negative       0.80      1.00      0.89         4
     neutral       0.75      1.00      0.86         3
    positive       1.00      0.60      0.75         5

    accuracy                           0.83        12
   macro avg       0.85      0.87      0.83        12
weighted avg       0.87      0.83      0.82        12


CLASSIFICATION REPORT FOR SVM
              precision    recall  f1-score   support

    negative       0.75      0.75      0.75         4
     neutral       0.75      1.00      0.86         3
    positive       0.75      0.60      0.67         5

    accuracy                           0.75        12
   macro avg       0.75      0.78      0.76        12
weighted avg       0.75      0.75      0.74        12



In [13]:
from sklearn.metrics import accuracy_score

# Calculate accuracy for TextBlob
tb_accuracy = accuracy_score(actual_labels, tb_predicted)

# Calculate accuracy for VADER
vader_accuracy = accuracy_score(actual_labels, vader_predicted)

# Calculate accuracy for Naive Bayes
nb_accuracy = accuracy_score(y_test, nb_predictions)

# Calculate accuracy for SVM
svm_accuracy = accuracy_score(y_test, svm_predictions)

# Display accuracy comparison
print("\n" + "="*60)
print("ACCURACY COMPARISON")
print("="*60)
print(f"{'Model':<25} {'Accuracy':<15}")
print("-"*40)
print(f"{'TextBlob (Lexicon-based)':<25} {tb_accuracy:.4f}")
print(f"{'VADER (Lexicon-based)':<25} {vader_accuracy:.4f}")
print(f"{'Naive Bayes (ML)':<25} {nb_accuracy:.4f}")
print(f"{'SVM (ML)':<25} {svm_accuracy:.4f}")


ACCURACY COMPARISON
Model                     Accuracy       
----------------------------------------
TextBlob (Lexicon-based)  0.5333
VADER (Lexicon-based)     0.6333
Naive Bayes (ML)          0.8333
SVM (ML)                  0.7500


In [14]:
print("\n" + "="*70)
print("LAB 7 - SENTIMENT ANALYSIS SUMMARY")
print("="*70)
print("""
INTERPRETATION OF RESULTS:

1. TEXTBLOB (Lexicon-based):
   - Uses pre-built dictionary of words with sentiment scores
   - Simple and fast but may miss context
   
2. VADER (Lexicon-based):
   - Specifically designed for social media text
   - Handles slang, emojis, and capitalization well
   
3. NAIVE BAYES (Machine Learning):
   - Learns patterns from training data
   - Generally performs well on text classification
   
4. SVM (Machine Learning):
   - Finds optimal hyperplane to separate classes
   - Effective for high-dimensional data

Based on the accuracy scores above:
- Best performing model: {} with {:.2f}% accuracy
- Machine learning approaches generally outperform lexicon-based
- VADER performs better than TextBlob for social media-style text
""".format(
    "Naive Bayes" if nb_accuracy >= svm_accuracy else "SVM",
    max(nb_accuracy, svm_accuracy) * 100
))


LAB 7 - SENTIMENT ANALYSIS SUMMARY

INTERPRETATION OF RESULTS:

1. TEXTBLOB (Lexicon-based):
   - Uses pre-built dictionary of words with sentiment scores
   - Simple and fast but may miss context

2. VADER (Lexicon-based):
   - Specifically designed for social media text
   - Handles slang, emojis, and capitalization well

3. NAIVE BAYES (Machine Learning):
   - Learns patterns from training data
   - Generally performs well on text classification

4. SVM (Machine Learning):
   - Finds optimal hyperplane to separate classes
   - Effective for high-dimensional data

Based on the accuracy scores above:
- Best performing model: Naive Bayes with 83.33% accuracy
- Machine learning approaches generally outperform lexicon-based
- VADER performs better than TextBlob for social media-style text

